In [1]:
import os

os.environ.pop("SPARK_HOME", None)
os.environ.pop("PYSPARK_SUBMIT_ARGS", None)
os.environ.pop("PYSPARK_PYTHON", None)
os.environ.pop("PYSPARK_DRIVER_PYTHON", None)

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Taxi Forecasting") \
    .config("spark.driver.memory", "2g") \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()

df_final = spark.read.parquet("df_final.parquet")
train_df = spark.read.parquet("train_df.parquet")
val_df = spark.read.parquet("val_df.parquet")
test_df = spark.read.parquet("test_df.parquet")

26/04/06 11:47:10 WARN Utils: Your hostname, Prashants-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.0.53 instead (on interface en0)
26/04/06 11:47:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/06 11:47:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
train_pd = train_df.toPandas()
val_pd = val_df.toPandas()
test_pd = test_df.toPandas()

In [3]:
print("df_final columns:", df_final.columns)
print("train_df columns:", train_df.columns)
print("val_df columns:", val_df.columns)
print("test_df columns:", test_df.columns)

df_final columns: ['lat_bin', 'lon_bin', 'pickup_date', 'pickup_hour', 'demand', 'avg_fare', 'avg_distance', 'long_trip_ratio', 'hour_str', 'datetime_str', 'datetime', 'lag_1', 'lag_2', 'lag_3', 'log_demand', 'rolling_3', 'datetime_num']
train_df columns: ['lat_bin', 'lon_bin', 'pickup_date', 'pickup_hour', 'demand', 'avg_fare', 'avg_distance', 'long_trip_ratio', 'hour_str', 'datetime_str', 'datetime', 'lag_1', 'lag_2', 'lag_3', 'log_demand', 'rolling_3', 'datetime_num']
val_df columns: ['lat_bin', 'lon_bin', 'pickup_date', 'pickup_hour', 'demand', 'avg_fare', 'avg_distance', 'long_trip_ratio', 'hour_str', 'datetime_str', 'datetime', 'lag_1', 'lag_2', 'lag_3', 'log_demand', 'rolling_3', 'datetime_num']
test_df columns: ['lat_bin', 'lon_bin', 'pickup_date', 'pickup_hour', 'demand', 'avg_fare', 'avg_distance', 'long_trip_ratio', 'hour_str', 'datetime_str', 'datetime', 'lag_1', 'lag_2', 'lag_3', 'log_demand', 'rolling_3', 'datetime_num']


In [4]:
df_final.printSchema()
train_df.printSchema()
val_df.printSchema()
test_df.printSchema()

root
 |-- lat_bin: long (nullable = true)
 |-- lon_bin: long (nullable = true)
 |-- pickup_date: date (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- demand: long (nullable = true)
 |-- avg_fare: double (nullable = true)
 |-- avg_distance: double (nullable = true)
 |-- long_trip_ratio: double (nullable = true)
 |-- hour_str: string (nullable = true)
 |-- datetime_str: string (nullable = true)
 |-- datetime: timestamp (nullable = true)
 |-- lag_1: long (nullable = true)
 |-- lag_2: long (nullable = true)
 |-- lag_3: long (nullable = true)
 |-- log_demand: double (nullable = true)
 |-- rolling_3: double (nullable = true)
 |-- datetime_num: long (nullable = true)

root
 |-- lat_bin: long (nullable = true)
 |-- lon_bin: long (nullable = true)
 |-- pickup_date: date (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- demand: long (nullable = true)
 |-- avg_fare: double (nullable = true)
 |-- avg_distance: double (nullable = true)
 |-- long_trip_ratio: doubl

In [5]:
set(df_final.columns) - set(train_df.columns)
set(df_final.columns) - set(val_df.columns)
set(df_final.columns) - set(test_df.columns)

set()

In [6]:
set(train_df.columns) - set(df_final.columns)

set()

In [7]:
base_cols = [
    'lat_bin',
    'lon_bin',
    'pickup_date',
    'pickup_hour',
    'demand',
    'avg_fare',
    'avg_distance',
    'long_trip_ratio'
]

df = df_final.select(base_cols)

In [8]:
print(df.columns)
df.show(5)

['lat_bin', 'lon_bin', 'pickup_date', 'pickup_hour', 'demand', 'avg_fare', 'avg_distance', 'long_trip_ratio']
+-------+-------+-----------+-----------+------+--------+------------+---------------+
|lat_bin|lon_bin|pickup_date|pickup_hour|demand|avg_fare|avg_distance|long_trip_ratio|
+-------+-------+-----------+-----------+------+--------+------------+---------------+
|   4054|  -7400| 2016-02-09|         20|     1|    11.0|         2.2|            0.0|
|   4054|  -7400| 2016-02-11|          2|     1|     6.0|        1.22|            0.0|
|   4054|  -7400| 2016-02-11|          5|     1|    12.0|        3.45|            0.0|
|   4054|  -7400| 2016-02-11|         12|     1|    35.0|        9.38|            0.0|
|   4054|  -7400| 2016-02-13|          1|     1|     5.0|        0.82|            0.0|
+-------+-------+-----------+-----------+------+--------+------------+---------------+
only showing top 5 rows



In [9]:
from pyspark.sql.functions import col, to_timestamp, concat_ws, lpad

df = df.withColumn(
    "pickup_hour_padded",
    lpad(col("pickup_hour").cast("string"), 2, "0")
)

df = df.withColumn(
    "timestamp",
    to_timestamp(concat_ws(" ", col("pickup_date"), col("pickup_hour_padded")))
)

In [10]:
df.select("pickup_date", "pickup_hour", "timestamp").show(5, False)

+-----------+-----------+-------------------+
|pickup_date|pickup_hour|timestamp          |
+-----------+-----------+-------------------+
|2016-02-09 |20         |2016-02-09 20:00:00|
|2016-02-11 |2          |2016-02-11 02:00:00|
|2016-02-11 |5          |2016-02-11 05:00:00|
|2016-02-11 |12         |2016-02-11 12:00:00|
|2016-02-13 |1          |2016-02-13 01:00:00|
+-----------+-----------+-------------------+
only showing top 5 rows



In [11]:
from pyspark.sql.functions import count

dup_check = df.groupBy("lat_bin", "lon_bin", "timestamp") \
    .agg(count("*").alias("cnt")) \
    .filter(col("cnt") > 1)

dup_count = dup_check.count()
print("Duplicate rows:", dup_count)

Duplicate rows: 0


In [12]:
df = df.orderBy("lat_bin", "lon_bin", "timestamp")

In [13]:
df.filter((col("lat_bin") == 4054) & (col("lon_bin") == -7400)) \
  .select("timestamp", "demand") \
  .show(20, False)



+-------------------+------+
|timestamp          |demand|
+-------------------+------+
|2016-02-09 20:00:00|1     |
|2016-02-11 02:00:00|1     |
|2016-02-11 05:00:00|1     |
|2016-02-11 12:00:00|1     |
|2016-02-13 01:00:00|1     |
|2016-02-13 15:00:00|1     |
|2016-02-18 19:00:00|1     |
|2016-02-23 06:00:00|1     |
+-------------------+------+



In [14]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag

window = Window.partitionBy("lat_bin", "lon_bin").orderBy("timestamp")

df_check = df.withColumn("prev_time", lag("timestamp").over(window))

df_check = df_check.withColumn(
    "time_diff_hours",
    (col("timestamp").cast("long") - col("prev_time").cast("long")) / 3600
)

df_check.filter(col("time_diff_hours") > 1).show()

+-------+-------+-----------+-----------+------+--------+------------+---------------+------------------+-------------------+-------------------+---------------+
|lat_bin|lon_bin|pickup_date|pickup_hour|demand|avg_fare|avg_distance|long_trip_ratio|pickup_hour_padded|          timestamp|          prev_time|time_diff_hours|
+-------+-------+-----------+-----------+------+--------+------------+---------------+------------------+-------------------+-------------------+---------------+
|   4054|  -7401| 2016-02-10|         23|     1|     8.0|        1.84|            0.0|                23|2016-02-10 23:00:00|2016-02-08 18:00:00|           53.0|
|   4054|  -7401| 2016-02-12|         23|     1|     6.0|        0.71|            0.0|                23|2016-02-12 23:00:00|2016-02-10 23:00:00|           48.0|
|   4054|  -7401| 2016-02-13|         20|     1|     9.5|        1.45|            0.0|                20|2016-02-13 20:00:00|2016-02-12 23:00:00|           21.0|
|   4054|  -7401| 2016-02-13

In [15]:
from pyspark.sql.functions import min, max

time_range = df.select(
    min("timestamp").alias("min_time"),
    max("timestamp").alias("max_time")
)

time_range.show()

+-------------------+-------------------+
|           min_time|           max_time|
+-------------------+-------------------+
|2015-01-01 04:00:00|2016-03-31 23:00:00|
+-------------------+-------------------+



In [16]:
from pyspark.sql.functions import sequence, explode, expr

time_df = time_range.select(
    explode(
        sequence(
            col("min_time"),
            col("max_time"),
            expr("interval 1 hour")
        )
    ).alias("timestamp")
)

In [17]:
time_df.show(5)
time_df.count()

+-------------------+
|          timestamp|
+-------------------+
|2015-01-01 04:00:00|
|2015-01-01 05:00:00|
|2015-01-01 06:00:00|
|2015-01-01 07:00:00|
|2015-01-01 08:00:00|
+-------------------+
only showing top 5 rows



10940

In [18]:
locations = df.select("lat_bin", "lon_bin").distinct()

In [19]:
locations.count()

948

In [20]:
full_grid = locations.crossJoin(time_df)
full_grid.count()

10371120

In [21]:
df_full = full_grid.join(df,on=["lat_bin", "lon_bin", "timestamp"],how="left")

In [22]:
from pyspark.sql.functions import coalesce, lit

df_full = df_full.withColumn(
    "demand",
    coalesce(col("demand"), lit(0))
)

In [23]:
df_full.filter((col("lat_bin") == 4054) & (col("lon_bin") == -7400)) \
    .select("timestamp", "demand") \
    .show(50, False)

+-------------------+------+
|timestamp          |demand|
+-------------------+------+
|2015-01-01 04:00:00|0     |
|2015-01-01 05:00:00|0     |
|2015-01-01 06:00:00|0     |
|2015-01-01 07:00:00|0     |
|2015-01-01 08:00:00|0     |
|2015-01-01 09:00:00|0     |
|2015-01-01 10:00:00|0     |
|2015-01-01 11:00:00|0     |
|2015-01-01 12:00:00|0     |
|2015-01-01 13:00:00|0     |
|2015-01-01 14:00:00|0     |
|2015-01-01 15:00:00|0     |
|2015-01-01 16:00:00|0     |
|2015-01-01 17:00:00|0     |
|2015-01-01 18:00:00|0     |
|2015-01-01 19:00:00|0     |
|2015-01-01 20:00:00|0     |
|2015-01-01 21:00:00|0     |
|2015-01-01 22:00:00|0     |
|2015-01-01 23:00:00|0     |
|2015-01-02 00:00:00|0     |
|2015-01-02 01:00:00|0     |
|2015-01-02 02:00:00|0     |
|2015-01-02 03:00:00|0     |
|2015-01-02 04:00:00|0     |
|2015-01-02 05:00:00|0     |
|2015-01-02 06:00:00|0     |
|2015-01-02 07:00:00|0     |
|2015-01-02 08:00:00|0     |
|2015-01-02 09:00:00|0     |
|2015-01-02 10:00:00|0     |
|2015-01-02 11

In [24]:
df_full.filter(col("demand") > 0).show(10)

+-------+-------+-------------------+-----------+-----------+------+--------+------------+---------------+------------------+
|lat_bin|lon_bin|          timestamp|pickup_date|pickup_hour|demand|avg_fare|avg_distance|long_trip_ratio|pickup_hour_padded|
+-------+-------+-------------------+-----------+-----------+------+--------+------------+---------------+------------------+
|   4054|  -7400|2016-02-09 20:00:00| 2016-02-09|         20|     1|    11.0|         2.2|            0.0|                20|
|   4054|  -7400|2016-02-11 02:00:00| 2016-02-11|          2|     1|     6.0|        1.22|            0.0|                02|
|   4054|  -7400|2016-02-11 05:00:00| 2016-02-11|          5|     1|    12.0|        3.45|            0.0|                05|
|   4054|  -7400|2016-02-11 12:00:00| 2016-02-11|         12|     1|    35.0|        9.38|            0.0|                12|
|   4054|  -7400|2016-02-13 01:00:00| 2016-02-13|          1|     1|     5.0|        0.82|            0.0|            

In [25]:
from pyspark.sql.window import Window

window = Window.partitionBy("lat_bin", "lon_bin").orderBy("timestamp")

from pyspark.sql.functions import lag

df_full = df_full.withColumn("lag_1", lag("demand", 1).over(window))
df_full = df_full.withColumn("lag_2", lag("demand", 2).over(window))
df_full = df_full.withColumn("lag_3", lag("demand", 3).over(window))

df_full.filter((col("lat_bin") == 4054) & (col("lon_bin") == -7400)) \
    .select("timestamp", "demand", "lag_1", "lag_2", "lag_3") \
    .show(20, False)

df_full.filter(
    (col("lat_bin") == 4054) &
    (col("lon_bin") == -7400) &
    (col("demand") > 0)
).select("timestamp", "demand").show(10, False)

df_full.groupBy("lat_bin", "lon_bin") \
    .agg({"demand": "sum"}) \
    .orderBy("sum(demand)", ascending=False) \
    .show(10)

+-------------------+------+-----+-----+-----+
|timestamp          |demand|lag_1|lag_2|lag_3|
+-------------------+------+-----+-----+-----+
|2015-01-01 04:00:00|0     |NULL |NULL |NULL |
|2015-01-01 05:00:00|0     |0    |NULL |NULL |
|2015-01-01 06:00:00|0     |0    |0    |NULL |
|2015-01-01 07:00:00|0     |0    |0    |0    |
|2015-01-01 08:00:00|0     |0    |0    |0    |
|2015-01-01 09:00:00|0     |0    |0    |0    |
|2015-01-01 10:00:00|0     |0    |0    |0    |
|2015-01-01 11:00:00|0     |0    |0    |0    |
|2015-01-01 12:00:00|0     |0    |0    |0    |
|2015-01-01 13:00:00|0     |0    |0    |0    |
|2015-01-01 14:00:00|0     |0    |0    |0    |
|2015-01-01 15:00:00|0     |0    |0    |0    |
|2015-01-01 16:00:00|0     |0    |0    |0    |
|2015-01-01 17:00:00|0     |0    |0    |0    |
|2015-01-01 18:00:00|0     |0    |0    |0    |
|2015-01-01 19:00:00|0     |0    |0    |0    |
|2015-01-01 20:00:00|0     |0    |0    |0    |
|2015-01-01 21:00:00|0     |0    |0    |0    |
|2015-01-01 2

In [26]:
from pyspark.sql.functions import avg
from pyspark.sql.window import Window

window_rolling = Window.partitionBy("lat_bin", "lon_bin") \
    .orderBy("timestamp") \
    .rowsBetween(-3, -1)

df_full = df_full.withColumn(
    "rolling_3",
    avg("demand").over(window_rolling)
)

df_full.filter((col("lat_bin") == 4054) & (col("lon_bin") == -7400)) \
    .select("timestamp", "demand", "lag_1", "rolling_3") \
    .show(20, False)

top_region = df_full.groupBy("lat_bin", "lon_bin") \
    .agg({"demand": "sum"}) \
    .orderBy("sum(demand)", ascending=False) \
    .first()

lat = top_region["lat_bin"]
lon = top_region["lon_bin"]

df_full.filter((col("lat_bin") == lat) & (col("lon_bin") == lon)) \
    .select("timestamp", "demand", "lag_1", "rolling_3") \
    .show(50, False)

+-------------------+------+-----+---------+
|timestamp          |demand|lag_1|rolling_3|
+-------------------+------+-----+---------+
|2015-01-01 04:00:00|0     |NULL |NULL     |
|2015-01-01 05:00:00|0     |0    |0.0      |
|2015-01-01 06:00:00|0     |0    |0.0      |
|2015-01-01 07:00:00|0     |0    |0.0      |
|2015-01-01 08:00:00|0     |0    |0.0      |
|2015-01-01 09:00:00|0     |0    |0.0      |
|2015-01-01 10:00:00|0     |0    |0.0      |
|2015-01-01 11:00:00|0     |0    |0.0      |
|2015-01-01 12:00:00|0     |0    |0.0      |
|2015-01-01 13:00:00|0     |0    |0.0      |
|2015-01-01 14:00:00|0     |0    |0.0      |
|2015-01-01 15:00:00|0     |0    |0.0      |
|2015-01-01 16:00:00|0     |0    |0.0      |
|2015-01-01 17:00:00|0     |0    |0.0      |
|2015-01-01 18:00:00|0     |0    |0.0      |
|2015-01-01 19:00:00|0     |0    |0.0      |
|2015-01-01 20:00:00|0     |0    |0.0      |
|2015-01-01 21:00:00|0     |0    |0.0      |
|2015-01-01 22:00:00|0     |0    |0.0      |
|2015-01-0

+-------------------+------+-----+------------------+
|timestamp          |demand|lag_1|rolling_3         |
+-------------------+------+-----+------------------+
|2015-01-01 04:00:00|369   |NULL |NULL              |
|2015-01-01 05:00:00|212   |369  |369.0             |
|2015-01-01 06:00:00|158   |212  |290.5             |
|2015-01-01 07:00:00|150   |158  |246.33333333333334|
|2015-01-01 08:00:00|202   |150  |173.33333333333334|
|2015-01-01 09:00:00|308   |202  |170.0             |
|2015-01-01 10:00:00|560   |308  |220.0             |
|2015-01-01 11:00:00|840   |560  |356.6666666666667 |
|2015-01-01 12:00:00|1073  |840  |569.3333333333334 |
|2015-01-01 13:00:00|1050  |1073 |824.3333333333334 |
|2015-01-01 14:00:00|1035  |1050 |987.6666666666666 |
|2015-01-01 15:00:00|950   |1035 |1052.6666666666667|
|2015-01-01 16:00:00|812   |950  |1011.6666666666666|
|2015-01-01 17:00:00|911   |812  |932.3333333333334 |
|2015-01-01 18:00:00|1040  |911  |891.0             |
|2015-01-01 19:00:00|957   |

In [27]:
from pyspark.sql.functions import hour, dayofweek, month

df_full = df_full.withColumn("hour", hour("timestamp"))
df_full = df_full.withColumn("day_of_week", dayofweek("timestamp"))
df_full = df_full.withColumn("month", month("timestamp"))

df_full.select("timestamp", "hour", "day_of_week", "month").show(10, False)

from pyspark.sql.functions import sin, cos
import math

df_full = df_full.withColumn("hour_sin", sin(2 * math.pi * col("hour") / 24))

df_full = df_full.withColumn("hour_cos", cos(2 * math.pi * col("hour") / 24))

df_full.select("hour", "hour_sin", "hour_cos").show(24)

+-------------------+----+-----------+-----+
|timestamp          |hour|day_of_week|month|
+-------------------+----+-----------+-----+
|2015-01-01 04:00:00|4   |5          |1    |
|2015-01-01 05:00:00|5   |5          |1    |
|2015-01-01 06:00:00|6   |5          |1    |
|2015-01-01 07:00:00|7   |5          |1    |
|2015-01-01 08:00:00|8   |5          |1    |
|2015-01-01 09:00:00|9   |5          |1    |
|2015-01-01 10:00:00|10  |5          |1    |
|2015-01-01 11:00:00|11  |5          |1    |
|2015-01-01 12:00:00|12  |5          |1    |
|2015-01-01 13:00:00|13  |5          |1    |
+-------------------+----+-----------+-----+
only showing top 10 rows

+----+--------------------+--------------------+
|hour|            hour_sin|            hour_cos|
+----+--------------------+--------------------+
|   4|  0.8660254037844386|  0.5000000000000001|
|   5|  0.9659258262890683| 0.25881904510252074|
|   6|                 1.0|6.123233995736766...|
|   7|  0.9659258262890683|-0.25881904510252063|
|

In [28]:
from pyspark.sql.window import Window
from pyspark.sql.functions import avg

# Define window for rolling calculation
window_rolling = Window.partitionBy("lat_bin", "lon_bin").orderBy("timestamp").rowsBetween(-3, -1)

df_full = df_full.drop("pickup_hour_padded")

# Add rolling_3 column
df_full = df_full.withColumn(
	"rolling_3",
	avg("demand").over(window_rolling)
)

df_model = df_full.dropna(subset=["lag_1", "lag_2", "lag_3", "rolling_3"])
df_model.select("lag_1", "lag_2", "lag_3", "rolling_3").show(5)

+-----+-----+-----+---------+
|lag_1|lag_2|lag_3|rolling_3|
+-----+-----+-----+---------+
|    0|    0|    0|      0.0|
|    0|    0|    0|      0.0|
|    0|    0|    0|      0.0|
|    0|    0|    0|      0.0|
|    0|    0|    0|      0.0|
+-----+-----+-----+---------+
only showing top 5 rows



In [29]:
df_full.filter(col("demand") == 0).count()

9875652

# Important 

In [30]:
df_model.columns

['lat_bin',
 'lon_bin',
 'timestamp',
 'pickup_date',
 'pickup_hour',
 'demand',
 'avg_fare',
 'avg_distance',
 'long_trip_ratio',
 'lag_1',
 'lag_2',
 'lag_3',
 'rolling_3',
 'hour',
 'day_of_week',
 'month',
 'hour_sin',
 'hour_cos']

In [31]:
df_model.filter(col("demand") == 0).count()

9873428

In [32]:
from pyspark.sql.functions import avg, stddev

# Location-level aggregates
location_stats = df_model.groupBy("lat_bin", "lon_bin").agg(avg("demand").alias("loc_avg_demand"),stddev("demand").alias("loc_std_demand"))

df_model = df_model.join(location_stats, on=["lat_bin", "lon_bin"], how="left")

In [33]:
from pyspark.sql.functions import when, col, hour

# Add hour column from timestamp
df_model = df_model.withColumn("hour", hour("timestamp"))
df_model = df_model.withColumn("is_peak_hour",when((col("hour").between(7, 10)) | (col("hour").between(17, 21)), 1).otherwise(0))
df_model = df_model.withColumn("is_weekend",when(col("day_of_week").isin([6,7]), 1).otherwise(0))
df_model = df_model.withColumn("is_night",when((col("hour") <= 5) | (col("hour") >= 22), 1).otherwise(0))


In [34]:
df_full.filter(col("demand") == 0).count()

9875652

In [35]:
from collections import Counter

Counter(df_model.columns)

Counter({'lat_bin': 1,
         'lon_bin': 1,
         'timestamp': 1,
         'pickup_date': 1,
         'pickup_hour': 1,
         'demand': 1,
         'avg_fare': 1,
         'avg_distance': 1,
         'long_trip_ratio': 1,
         'lag_1': 1,
         'lag_2': 1,
         'lag_3': 1,
         'rolling_3': 1,
         'hour': 1,
         'day_of_week': 1,
         'month': 1,
         'hour_sin': 1,
         'hour_cos': 1,
         'loc_avg_demand': 1,
         'loc_std_demand': 1,
         'is_peak_hour': 1,
         'is_weekend': 1,
         'is_night': 1})

In [36]:
df_model = df_model.withColumn("demand_vs_avg",col("lag_1") / (col("loc_avg_demand") + 1e-5))

df_model = df_model.withColumn("demand_zscore",(col("lag_1") - col("loc_avg_demand")) / (col("loc_std_demand") + 1e-5))

In [37]:
from pyspark.sql.window import Window
from pyspark.sql.functions import avg

window_spec = Window.partitionBy("lat_bin", "lon_bin").orderBy("timestamp").rowsBetween(-6, 0)

df_model = df_model.withColumn(
    "rolling_6",
    avg("demand").over(window_spec)
)

In [38]:
df_model.columns 

['lat_bin',
 'lon_bin',
 'timestamp',
 'pickup_date',
 'pickup_hour',
 'demand',
 'avg_fare',
 'avg_distance',
 'long_trip_ratio',
 'lag_1',
 'lag_2',
 'lag_3',
 'rolling_3',
 'hour',
 'day_of_week',
 'month',
 'hour_sin',
 'hour_cos',
 'loc_avg_demand',
 'loc_std_demand',
 'is_peak_hour',
 'is_weekend',
 'is_night',
 'demand_vs_avg',
 'demand_zscore',
 'rolling_6']

In [39]:
df_model.printSchema()

root
 |-- lat_bin: long (nullable = true)
 |-- lon_bin: long (nullable = true)
 |-- timestamp: timestamp (nullable = false)
 |-- pickup_date: date (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- demand: long (nullable = false)
 |-- avg_fare: double (nullable = true)
 |-- avg_distance: double (nullable = true)
 |-- long_trip_ratio: double (nullable = true)
 |-- lag_1: long (nullable = true)
 |-- lag_2: long (nullable = true)
 |-- lag_3: long (nullable = true)
 |-- rolling_3: double (nullable = true)
 |-- hour: integer (nullable = false)
 |-- day_of_week: integer (nullable = false)
 |-- month: integer (nullable = false)
 |-- hour_sin: double (nullable = true)
 |-- hour_cos: double (nullable = true)
 |-- loc_avg_demand: double (nullable = true)
 |-- loc_std_demand: double (nullable = true)
 |-- is_peak_hour: integer (nullable = false)
 |-- is_weekend: integer (nullable = false)
 |-- is_night: integer (nullable = false)
 |-- demand_vs_avg: double (nullable = true)
 |-- d

In [40]:
from pyspark.sql.functions import count, isnan

df_model.select([
    count(when(isnan(c), c)).alias(c)
    for c in ["loc_avg_demand", "demand_vs_avg", "rolling_6"]
]).show()

+--------------+-------------+---------+
|loc_avg_demand|demand_vs_avg|rolling_6|
+--------------+-------------+---------+
|             0|            0|        0|
+--------------+-------------+---------+



In [41]:
df_model.filter(col("demand") == 0).count()

9873428

In [42]:
df_model.select("loc_avg_demand").describe().show()
df_model.select("demand_vs_avg").describe().show()

+-------+--------------------+
|summary|      loc_avg_demand|
+-------+--------------------+
|  count|            10368276|
|   mean|   4.421331376699482|
| stddev|  23.077340124318095|
|    min|9.143275121148396E-5|
|    max|   260.9889366371034|
+-------+--------------------+



+-------+------------------+
|summary|     demand_vs_avg|
+-------+------------------+
|  count|          10368276|
|   mean|0.9877771990469532|
| stddev| 34.65482040417263|
|    min|               0.0|
|    max|10552.298375555176|
+-------+------------------+



In [43]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag

window_spec = Window.partitionBy("lat_bin", "lon_bin").orderBy("timestamp")

df_model = df_model.withColumn("lag_24",lag("demand", 24).over(window_spec))

df_model = df_model.withColumn("lag_168",lag("demand", 168).over(window_spec))

In [44]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, col

window_spec = Window.partitionBy("lat_bin", "lon_bin").orderBy("timestamp")

df_model = df_model.withColumn("lag_3",lag("demand", 3).over(window_spec))

In [45]:
df_model = df_model.withColumn("trend_1_3",col("lag_1") - col("lag_3"))

df_model = df_model.withColumn("trend_1_6",col("lag_1") - col("rolling_6"))

In [46]:
from pyspark.sql.functions import when

df_model = df_model.withColumn("is_active_zone",when(col("loc_avg_demand") > 2, 1).otherwise(0))

In [47]:
df_model = df_model.withColumn("peak_intensity",col("is_peak_hour") * col("loc_avg_demand"))

In [48]:
from pyspark.sql.functions import log1p

df_model = df_model.withColumn("log_lag_1",log1p(col("lag_1")))

df_model = df_model.withColumn("log_rolling_6",log1p(col("rolling_6")))

In [49]:
df_model = df_model.withColumn("hour_x_zone",col("hour") * col("loc_avg_demand"))

In [50]:
df_model.select("lag_1", "lag_3", "rolling_6").show(5)

+-----+-----+---------+
|lag_1|lag_3|rolling_6|
+-----+-----+---------+
|    0| NULL|      0.0|
|    0| NULL|      0.0|
|    0| NULL|      0.0|
|    0|    0|      0.0|
|    0|    0|      0.0|
+-----+-----+---------+
only showing top 5 rows



In [51]:
df_model = df_model.dropna(subset=["lag_3", "lag_24", "lag_168"])

In [52]:
df_model.filter(col("demand") == 0).count()

9741937

In [53]:
df_model.select("lag_1", "lag_3", "rolling_6","trend_1_3", "trend_1_6").show(10)

+-----+-----+---------+---------+---------+
|lag_1|lag_3|rolling_6|trend_1_3|trend_1_6|
+-----+-----+---------+---------+---------+
|    0|    0|      0.0|        0|      0.0|
|    0|    0|      0.0|        0|      0.0|
|    0|    0|      0.0|        0|      0.0|
|    0|    0|      0.0|        0|      0.0|
|    0|    0|      0.0|        0|      0.0|
|    0|    0|      0.0|        0|      0.0|
|    0|    0|      0.0|        0|      0.0|
|    0|    0|      0.0|        0|      0.0|
|    0|    0|      0.0|        0|      0.0|
|    0|    0|      0.0|        0|      0.0|
+-----+-----+---------+---------+---------+
only showing top 10 rows



In [54]:

df_model = df_model.withColumn("recent_activity",(col("lag_1") + col("lag_3") + col("rolling_6")))

In [55]:
from pyspark.sql.functions import log1p, col

df_model = df_model.withColumn(
    "log_demand",
    log1p(col("demand"))
)

In [56]:
df_model.filter(col("demand") == 0).count()

9741937

In [57]:
df_model = df_model.withColumn(
    "persistence",
    (col("lag_1") > 0).cast("int") +
    (col("lag_3") > 0).cast("int") +
    (col("lag_24") > 0).cast("int")
)

In [58]:
df_model.filter(col("demand") == 0).count()

9741937

In [59]:
print(type(df_model))

<class 'pyspark.sql.dataframe.DataFrame'>


In [60]:
from pyspark.sql.functions import when, col

df_model = df_model.withColumn(
    "has_demand",
    when(col("demand") > 0, 1).otherwise(0)
)

In [61]:
df_model.filter(col("demand") == 0).count()

9741937

In [62]:
from pyspark.sql.functions import col

In [66]:
df_model.columns

['lat_bin',
 'lon_bin',
 'timestamp',
 'pickup_date',
 'pickup_hour',
 'demand',
 'avg_fare',
 'avg_distance',
 'long_trip_ratio',
 'lag_1',
 'lag_2',
 'lag_3',
 'rolling_3',
 'hour',
 'day_of_week',
 'month',
 'hour_sin',
 'hour_cos',
 'loc_avg_demand',
 'loc_std_demand',
 'is_peak_hour',
 'is_weekend',
 'is_night',
 'demand_vs_avg',
 'demand_zscore',
 'rolling_6',
 'lag_24',
 'lag_168',
 'trend_1_3',
 'trend_1_6',
 'is_active_zone',
 'peak_intensity',
 'log_lag_1',
 'log_rolling_6',
 'hour_x_zone',
 'recent_activity',
 'log_demand',
 'persistence',
 'has_demand']

In [68]:
print("Rows:", df_model.count())
print("Columns:", len(df_model.columns))

Rows: 10209012
Columns: 39


In [65]:
df_model.columns 
df_model.printSchema()

root
 |-- lat_bin: long (nullable = true)
 |-- lon_bin: long (nullable = true)
 |-- timestamp: timestamp (nullable = false)
 |-- pickup_date: date (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- demand: long (nullable = false)
 |-- avg_fare: double (nullable = true)
 |-- avg_distance: double (nullable = true)
 |-- long_trip_ratio: double (nullable = true)
 |-- lag_1: long (nullable = true)
 |-- lag_2: long (nullable = true)
 |-- lag_3: long (nullable = true)
 |-- rolling_3: double (nullable = true)
 |-- hour: integer (nullable = false)
 |-- day_of_week: integer (nullable = false)
 |-- month: integer (nullable = false)
 |-- hour_sin: double (nullable = true)
 |-- hour_cos: double (nullable = true)
 |-- loc_avg_demand: double (nullable = true)
 |-- loc_std_demand: double (nullable = true)
 |-- is_peak_hour: integer (nullable = false)
 |-- is_weekend: integer (nullable = false)
 |-- is_night: integer (nullable = false)
 |-- demand_vs_avg: double (nullable = true)
 |-- d

In [69]:
print("Before save:", df_model.count())

Before save: 10209012


In [70]:
df_model.repartition(200, "pickup_date") \
    .write \
    .mode("overwrite") \
    .partitionBy("pickup_date") \
    .parquet("yellow_taxi/data/df_model_full")

26/04/06 11:57:40 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/06 11:57:42 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/06 11:57:43 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


In [72]:
df_model.select("timestamp").summary().show()

+-------+
|summary|
+-------+
|  count|
|   mean|
| stddev|
|    min|
|    25%|
|    50%|
|    75%|
|    max|
+-------+



In [73]:
from pyspark.sql.functions import year, month

df_model.groupBy(
    year("timestamp").alias("year"),
    month("timestamp").alias("month")
).count().orderBy("year", "month").show(50, False)

+----+-----+------+
|year|month|count |
+----+-----+------+
|2015|1    |539412|
|2015|2    |637056|
|2015|3    |705312|
|2015|4    |682560|
|2015|5    |705312|
|2015|6    |682560|
|2015|7    |705312|
|2015|8    |705312|
|2015|9    |682560|
|2015|10   |705312|
|2015|11   |682560|
|2015|12   |705312|
|2016|1    |705312|
|2016|2    |659808|
|2016|3    |705312|
+----+-----+------+



26/04/06 16:50:04 WARN TransportChannelHandler: Exception in connection from /192.168.0.53:55781
java.io.IOException: Operation timed out
	at java.base/sun.nio.ch.FileDispatcherImpl.read0(Native Method)
	at java.base/sun.nio.ch.SocketDispatcher.read(SocketDispatcher.java:39)
	at java.base/sun.nio.ch.IOUtil.readIntoNativeBuffer(IOUtil.java:276)
	at java.base/sun.nio.ch.IOUtil.read(IOUtil.java:233)
	at java.base/sun.nio.ch.IOUtil.read(IOUtil.java:223)
	at java.base/sun.nio.ch.SocketChannelImpl.read(SocketChannelImpl.java:356)
	at io.netty.buffer.PooledByteBuf.setBytes(PooledByteBuf.java:254)
	at io.netty.buffer.AbstractByteBuf.writeBytes(AbstractByteBuf.java:1132)
	at io.netty.channel.socket.nio.NioSocketChannel.doReadBytes(NioSocketChannel.java:357)
	at io.netty.channel.nio.AbstractNioByteChannel$NioByteUnsafe.read(AbstractNioByteChannel.java:151)
	at io.netty.channel.nio.NioEventLoop.processSelectedKey(NioEventLoop.java:788)
	at io.netty.channel.nio.NioEventLoop.processSelectedKeysOpti

In [71]:
df_check = spark.read.parquet("yellow_taxi/data/df_model_full")
print("After save:", df_check.count())

After save: 10209012


In [63]:
df_pos = df_model.filter("has_demand = 1")

df_neg = df_model.filter("has_demand = 0") \
                 .sample(fraction=0.3, seed=42)

df_model_balanced = df_pos.unionByName(df_neg)

In [64]:
df_model_balanced.filter(col("demand") == 0).count()

2925195

In [65]:
df_model.columns

['lat_bin',
 'lon_bin',
 'timestamp',
 'pickup_date',
 'pickup_hour',
 'demand',
 'avg_fare',
 'avg_distance',
 'long_trip_ratio',
 'lag_1',
 'lag_2',
 'lag_3',
 'rolling_3',
 'hour',
 'day_of_week',
 'month',
 'hour_sin',
 'hour_cos',
 'loc_avg_demand',
 'loc_std_demand',
 'is_peak_hour',
 'is_weekend',
 'is_night',
 'demand_vs_avg',
 'demand_zscore',
 'rolling_6',
 'lag_24',
 'lag_168',
 'trend_1_3',
 'trend_1_6',
 'is_active_zone',
 'peak_intensity',
 'log_lag_1',
 'log_rolling_6',
 'hour_x_zone',
 'recent_activity',
 'log_demand',
 'persistence',
 'has_demand']

In [66]:
df_model_balanced.columns

['lat_bin',
 'lon_bin',
 'timestamp',
 'pickup_date',
 'pickup_hour',
 'demand',
 'avg_fare',
 'avg_distance',
 'long_trip_ratio',
 'lag_1',
 'lag_2',
 'lag_3',
 'rolling_3',
 'hour',
 'day_of_week',
 'month',
 'hour_sin',
 'hour_cos',
 'loc_avg_demand',
 'loc_std_demand',
 'is_peak_hour',
 'is_weekend',
 'is_night',
 'demand_vs_avg',
 'demand_zscore',
 'rolling_6',
 'lag_24',
 'lag_168',
 'trend_1_3',
 'trend_1_6',
 'is_active_zone',
 'peak_intensity',
 'log_lag_1',
 'log_rolling_6',
 'hour_x_zone',
 'recent_activity',
 'log_demand',
 'persistence',
 'has_demand']

In [67]:
df_model_balanced.orderBy("loc_avg_demand", ascending=False).show(5)

26/04/05 21:40:35 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/05 21:40:36 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/05 21:40:36 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/05 21:40:37 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/05 21:40:37 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/05 21:40:37 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/05 21:40:37 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/05 21:40:38 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but re

+-------+-------+-------------------+-----------+-----------+------+------------------+------------------+--------------------+-----+-----+-----+-----------------+----+-----------+-----+--------------------+--------------------+-----------------+----------------+------------+----------+--------+-----------------+--------------------+------------------+------+-------+---------+-------------------+--------------+-----------------+-----------------+------------------+------------------+------------------+-----------------+-----------+----------+
|lat_bin|lon_bin|          timestamp|pickup_date|pickup_hour|demand|          avg_fare|      avg_distance|     long_trip_ratio|lag_1|lag_2|lag_3|        rolling_3|hour|day_of_week|month|            hour_sin|            hour_cos|   loc_avg_demand|  loc_std_demand|is_peak_hour|is_weekend|is_night|    demand_vs_avg|       demand_zscore|         rolling_6|lag_24|lag_168|trend_1_3|          trend_1_6|is_active_zone|   peak_intensity|        log_lag_1| 

In [68]:
df_model_balanced.filter(col("demand") == 0).count()

2925195

In [69]:
print("Before:", df_full.count())
print("After:", df_model_balanced.count())

Before: 10371120


26/04/05 21:41:10 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/05 21:41:10 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


After: 3392270


In [70]:
from pyspark.sql.functions import col

df_model_balanced = df_model_balanced.select('lat_bin',
 'lon_bin',
 'timestamp',
 'pickup_date',
 'pickup_hour',
 'demand',
 'avg_fare',
 'avg_distance',
 'long_trip_ratio',
 'lag_1',
 'lag_2',
 'lag_3',
 'rolling_3',
 'hour',
 'day_of_week',
 'month',
 'hour_sin',
 'hour_cos',
 'loc_avg_demand',
 'loc_std_demand',
 'is_peak_hour',
 'is_weekend',
 'is_night',
 'demand_vs_avg',
 'demand_zscore',
 'rolling_6',
 'lag_24',
 'lag_168',
 'trend_1_3',
 'trend_1_6',
 'is_active_zone',
 'peak_intensity',
 'log_lag_1',
 'log_rolling_6',
 'hour_x_zone',
 'recent_activity',
 'log_demand',
 'persistence',
 'has_demand')

In [71]:
df_model_balanced.select("demand").show(5)

+------+
|demand|
+------+
|     1|
|     1|
|     1|
|     1|
|     1|
+------+
only showing top 5 rows



In [72]:
from pyspark.sql.functions import skewness, col

df_model_balanced.agg(
    skewness(col("demand")).alias("raw_skew"),
    skewness(col("log_demand")).alias("log_skew")
).show()

26/04/05 21:41:18 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/05 21:41:18 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


+-----------------+-----------------+
|         raw_skew|         log_skew|
+-----------------+-----------------+
|10.29197247593732|3.951631664545382|
+-----------------+-----------------+



In [73]:
from pyspark.sql.functions import when

df_model_balanced= df_model_balanced.withColumn("has_demand",when(col("demand") > 0, 1).otherwise(0))

In [74]:
df_model_balanced.filter(col("demand") == 0).count()

2925195

In [75]:
df_model_balanced = df_model_balanced.withColumn("weight",when(col("demand") > 0, 2.0).otherwise(1.0))

In [76]:
df_model_balanced.columns

['lat_bin',
 'lon_bin',
 'timestamp',
 'pickup_date',
 'pickup_hour',
 'demand',
 'avg_fare',
 'avg_distance',
 'long_trip_ratio',
 'lag_1',
 'lag_2',
 'lag_3',
 'rolling_3',
 'hour',
 'day_of_week',
 'month',
 'hour_sin',
 'hour_cos',
 'loc_avg_demand',
 'loc_std_demand',
 'is_peak_hour',
 'is_weekend',
 'is_night',
 'demand_vs_avg',
 'demand_zscore',
 'rolling_6',
 'lag_24',
 'lag_168',
 'trend_1_3',
 'trend_1_6',
 'is_active_zone',
 'peak_intensity',
 'log_lag_1',
 'log_rolling_6',
 'hour_x_zone',
 'recent_activity',
 'log_demand',
 'persistence',
 'has_demand',
 'weight']

In [77]:
len(df_model_balanced.columns), len(set(df_model_balanced.columns))

(40, 40)

In [78]:
from pyspark.sql.functions import col, log1p, when

# log transform
df_model_balanced = df_model_balanced.withColumn("log_demand",log1p(col("demand")))

# classification label
df_model_balanced = df_model_balanced.withColumn("has_demand",when(col("demand") > 0, 1).otherwise(0))

# class weight (for imbalance)
df_model_balanced = df_model_balanced.withColumn("weight",when(col("demand") > 0, 2.0).otherwise(1.0))

In [79]:
df_model_balanced.show(20,False)

26/04/05 21:41:29 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/05 21:41:31 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


+-------+-------+-------------------+-----------+-----------+------+--------+------------+---------------+-----+-----+-----+------------------+----+-----------+-----+--------------------+-----------------------+---------------------+--------------------+------------+----------+--------+------------------+---------------------+-------------------+------+-------+---------+--------------------+--------------+---------------------+------------------+-------------------+---------------------+-------------------+------------------+-----------+----------+------+
|lat_bin|lon_bin|timestamp          |pickup_date|pickup_hour|demand|avg_fare|avg_distance|long_trip_ratio|lag_1|lag_2|lag_3|rolling_3         |hour|day_of_week|month|hour_sin            |hour_cos               |loc_avg_demand       |loc_std_demand      |is_peak_hour|is_weekend|is_night|demand_vs_avg     |demand_zscore        |rolling_6          |lag_24|lag_168|trend_1_3|trend_1_6           |is_active_zone|peak_intensity       |log_lag

In [80]:
df_model_balanced.filter(col("demand") == 0).count()

2925195

In [81]:
df_model_balanced.groupBy("has_demand").count().show()

26/04/05 21:41:53 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/05 21:41:53 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


+----------+-------+
|has_demand|  count|
+----------+-------+
|         1| 467075|
|         0|2925195|
+----------+-------+



In [82]:
from pyspark.sql.functions import count, when

df_model_balanced.select(
    (count(when(col("demand") == 0, True)) / count("*")).alias("zero_ratio"),
    (count(when(col("demand") > 0, True)) / count("*")).alias("non_zero_ratio")
).show()

26/04/05 21:42:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/05 21:42:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


+------------------+-------------------+
|        zero_ratio|     non_zero_ratio|
+------------------+-------------------+
|0.8623119621964054|0.13768803780359465|
+------------------+-------------------+



In [83]:
df_model_balanced.filter(col("demand") == 0).count()

2925195

In [84]:
df_model_balanced = df_model_balanced.dropna(
    subset=["lag_1", "lag_3", "lag_24", "rolling_3"]
)

In [85]:
from pyspark.sql.functions import min, max

df_model_balanced.select(min("timestamp"), max("timestamp")).show()

26/04/05 21:42:17 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/05 21:42:17 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


+-------------------+-------------------+
|     min(timestamp)|     max(timestamp)|
+-------------------+-------------------+
|2015-01-08 07:00:00|2016-03-31 23:00:00|
+-------------------+-------------------+



In [86]:
df_model_balanced.filter(col("demand") == 0).count()

26/04/05 21:42:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/05 21:42:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


2925195

In [87]:
from pyspark.sql.functions import col

# Get timestamps
min_ts = df_model_balanced.selectExpr("min(timestamp)").collect()[0][0]
max_ts = df_model_balanced.selectExpr("max(timestamp)").collect()[0][0]

# Convert to Python datetime
import pandas as pd

min_ts = pd.to_datetime(min_ts)
max_ts = pd.to_datetime(max_ts)

# Create splits
train_end = min_ts + (max_ts - min_ts) * 0.7
val_end   = min_ts + (max_ts - min_ts) * 0.85

print(train_end, val_end)

26/04/05 21:42:30 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/05 21:42:30 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/05 21:42:37 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/05 21:42:37 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


2015-11-18 08:36:00 2016-01-24 15:48:00


In [88]:
df_model_balanced.filter(col("demand") == 0).count()

26/04/05 21:42:43 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/05 21:42:43 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


2925195

In [89]:
train_end = train_end.floor('H')
val_end   = val_end.floor('H')

/var/folders/s4/m8r6vx7j7873dljxvwf3ts9r0000gn/T/ipykernel_39336/114655569.py:1: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  train_end = train_end.floor('H')
/var/folders/s4/m8r6vx7j7873dljxvwf3ts9r0000gn/T/ipykernel_39336/114655569.py:2: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  val_end   = val_end.floor('H')


In [90]:
df_model_balanced.columns 

['lat_bin',
 'lon_bin',
 'timestamp',
 'pickup_date',
 'pickup_hour',
 'demand',
 'avg_fare',
 'avg_distance',
 'long_trip_ratio',
 'lag_1',
 'lag_2',
 'lag_3',
 'rolling_3',
 'hour',
 'day_of_week',
 'month',
 'hour_sin',
 'hour_cos',
 'loc_avg_demand',
 'loc_std_demand',
 'is_peak_hour',
 'is_weekend',
 'is_night',
 'demand_vs_avg',
 'demand_zscore',
 'rolling_6',
 'lag_24',
 'lag_168',
 'trend_1_3',
 'trend_1_6',
 'is_active_zone',
 'peak_intensity',
 'log_lag_1',
 'log_rolling_6',
 'hour_x_zone',
 'recent_activity',
 'log_demand',
 'persistence',
 'has_demand',
 'weight']

In [91]:
train_df = df_model_balanced.filter(col("timestamp") <= train_end)

val_df = df_model_balanced.filter(
    (col("timestamp") > train_end) & 
    (col("timestamp") <= val_end)
)

test_df = df_model_balanced.filter(col("timestamp") > val_end)

In [92]:
print("Train:", train_df.count())
print("Val:", val_df.count())
print("Test:", test_df.count())

26/04/05 21:42:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/05 21:42:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


Train: 2213197


26/04/05 21:42:59 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/05 21:42:59 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


Val: 524262


26/04/05 21:43:06 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/05 21:43:06 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


Test: 654811


In [93]:
train_df.select("timestamp").orderBy("timestamp").show(3)
train_df.select("timestamp").orderBy(col("timestamp").desc()).show(3)

val_df.select("timestamp").orderBy("timestamp").show(3)
test_df.select("timestamp").orderBy("timestamp").show(3)

26/04/05 21:43:13 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/05 21:43:13 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


+-------------------+
|          timestamp|
+-------------------+
|2015-01-08 07:00:00|
|2015-01-08 07:00:00|
|2015-01-08 07:00:00|
+-------------------+
only showing top 3 rows



26/04/05 21:43:20 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/05 21:43:20 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


+-------------------+
|          timestamp|
+-------------------+
|2015-11-18 08:00:00|
|2015-11-18 08:00:00|
|2015-11-18 08:00:00|
+-------------------+
only showing top 3 rows



26/04/05 21:43:26 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/05 21:43:26 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


+-------------------+
|          timestamp|
+-------------------+
|2015-11-18 09:00:00|
|2015-11-18 09:00:00|
|2015-11-18 09:00:00|
+-------------------+
only showing top 3 rows



26/04/05 21:43:33 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/05 21:43:33 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


+-------------------+
|          timestamp|
+-------------------+
|2016-01-24 16:00:00|
|2016-01-24 16:00:00|
|2016-01-24 16:00:00|
+-------------------+
only showing top 3 rows



In [94]:
fill_dict = {
    "lag_1": 0,
    "lag_2": 0,
    "lag_3": 0,
    "lag_24": 0,
    "lag_168": 0,
    "rolling_3": 0,
    "rolling_6": 0,
    "trend_1_3": 0,
    "trend_1_6": 0,
    "loc_avg_demand": 0,
    "loc_std_demand": 0
}

train_df = train_df.fillna(fill_dict)
val_df   = val_df.fillna(fill_dict)
test_df  = test_df.fillna(fill_dict)

In [95]:
train_df.columns

['lat_bin',
 'lon_bin',
 'timestamp',
 'pickup_date',
 'pickup_hour',
 'demand',
 'avg_fare',
 'avg_distance',
 'long_trip_ratio',
 'lag_1',
 'lag_2',
 'lag_3',
 'rolling_3',
 'hour',
 'day_of_week',
 'month',
 'hour_sin',
 'hour_cos',
 'loc_avg_demand',
 'loc_std_demand',
 'is_peak_hour',
 'is_weekend',
 'is_night',
 'demand_vs_avg',
 'demand_zscore',
 'rolling_6',
 'lag_24',
 'lag_168',
 'trend_1_3',
 'trend_1_6',
 'is_active_zone',
 'peak_intensity',
 'log_lag_1',
 'log_rolling_6',
 'hour_x_zone',
 'recent_activity',
 'log_demand',
 'persistence',
 'has_demand',
 'weight']

In [96]:
val_df.columns

['lat_bin',
 'lon_bin',
 'timestamp',
 'pickup_date',
 'pickup_hour',
 'demand',
 'avg_fare',
 'avg_distance',
 'long_trip_ratio',
 'lag_1',
 'lag_2',
 'lag_3',
 'rolling_3',
 'hour',
 'day_of_week',
 'month',
 'hour_sin',
 'hour_cos',
 'loc_avg_demand',
 'loc_std_demand',
 'is_peak_hour',
 'is_weekend',
 'is_night',
 'demand_vs_avg',
 'demand_zscore',
 'rolling_6',
 'lag_24',
 'lag_168',
 'trend_1_3',
 'trend_1_6',
 'is_active_zone',
 'peak_intensity',
 'log_lag_1',
 'log_rolling_6',
 'hour_x_zone',
 'recent_activity',
 'log_demand',
 'persistence',
 'has_demand',
 'weight']

In [97]:
test_df.columns 

['lat_bin',
 'lon_bin',
 'timestamp',
 'pickup_date',
 'pickup_hour',
 'demand',
 'avg_fare',
 'avg_distance',
 'long_trip_ratio',
 'lag_1',
 'lag_2',
 'lag_3',
 'rolling_3',
 'hour',
 'day_of_week',
 'month',
 'hour_sin',
 'hour_cos',
 'loc_avg_demand',
 'loc_std_demand',
 'is_peak_hour',
 'is_weekend',
 'is_night',
 'demand_vs_avg',
 'demand_zscore',
 'rolling_6',
 'lag_24',
 'lag_168',
 'trend_1_3',
 'trend_1_6',
 'is_active_zone',
 'peak_intensity',
 'log_lag_1',
 'log_rolling_6',
 'hour_x_zone',
 'recent_activity',
 'log_demand',
 'persistence',
 'has_demand',
 'weight']

In [98]:
cls_features = ['lat_bin',
 'lon_bin',
 'timestamp',
 'pickup_date',
 'pickup_hour',
 'demand',
 'avg_fare',
 'avg_distance',
 'long_trip_ratio',
 'lag_1',
 'lag_2',
 'lag_3',
 'rolling_3',
 'hour',
 'day_of_week',
 'month',
 'hour_sin',
 'hour_cos',
 'loc_avg_demand',
 'loc_std_demand',
 'is_peak_hour',
 'is_weekend',
 'is_night',
 'demand_vs_avg',
 'demand_zscore',
 'rolling_6',
 'lag_24',
 'lag_168',
 'trend_1_3',
 'trend_1_6',
 'is_active_zone',
 'peak_intensity',
 'log_lag_1',
 'log_rolling_6',
 'hour_x_zone',
 'recent_activity',
 'log_demand',
 'persistence',
 'has_demand']

cls_features = [c for c in cls_features if c in train_df.columns]

train_cls = train_df.select(*cls_features, "has_demand", "weight")
val_cls   = val_df.select(*cls_features, "has_demand", "weight")
test_cls  = test_df.select(*cls_features, "has_demand", "weight")

In [100]:
# remove duplicate columns safely

def remove_duplicate_columns(df):
    cols = df.columns
    seen = set()
    unique_cols = []
    
    for c in cols:
        if c not in seen:
            seen.add(c)
            unique_cols.append(c)
    
    return df.select(*unique_cols)

train_cls = remove_duplicate_columns(train_cls)
val_cls   = remove_duplicate_columns(val_cls)
test_cls  = remove_duplicate_columns(test_cls)

In [ ]:
# Save to a clean modeling layer (NOT artifacts)

base_path = "yellow_taxi/model_input"

train_cls.write.mode("overwrite").parquet(f"{base_path}/train.parquet")
val_cls.write.mode("overwrite").parquet(f"{base_path}/val.parquet")
test_cls.write.mode("overwrite").parquet(f"{base_path}/test.parquet")

26/04/05 21:45:42 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/05 21:45:42 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/05 21:45:43 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/05 21:45:43 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/05 21:45:44 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/05 21:45:44 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/05 21:46:08 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/05 21:46:08 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/05 21:46:09 WARN RowBasedKeyValueBatch: Calling spill() on

26/04/06 05:52:44 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 954598 ms exceeds timeout 120000 ms
26/04/06 05:52:44 WARN SparkContext: Killing executors is not supported by current scheduler.
26/04/06 06:01:58 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:80)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:642)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1223)
	at o

In [ ]:
df_model_balanced = df_model_balanced.persist()
df_model_balanced.count()

3388585

In [ ]:
train_cls = train_cls.cache()
train_cls.count()  # force cache

train_cls.cache()
val_cls.cache()
test_cls.cache()

26/04/05 20:53:43 WARN CacheManager: Asked to cache already cached data.        


DataFrame[lat_bin: bigint, lon_bin: bigint, timestamp: timestamp, pickup_date: date, pickup_hour: int, demand: bigint, avg_fare: double, avg_distance: double, long_trip_ratio: double, lag_1: bigint, lag_2: bigint, lag_3: bigint, rolling_3: double, hour: int, day_of_week: int, month: int, hour_sin: double, hour_cos: double, loc_avg_demand: double, loc_std_demand: double, is_peak_hour: int, is_weekend: int, is_night: int, demand_vs_avg: double, demand_zscore: double, rolling_6: double, lag_24: bigint, lag_168: bigint, trend_1_3: bigint, trend_1_6: double, is_active_zone: int, peak_intensity: double, log_lag_1: double, log_rolling_6: double, hour_x_zone: double, recent_activity: double, log_demand: double, persistence: int, has_demand: int, has_demand: int, weight: double]

In [ ]:
train_cls.filter(col("demand") == 0).count()

2113747

In [ ]:
from pyspark.sql.functions import sum, col

train_cls.select(
    sum(col("has_demand")).alias("positive_cases"),
    (count("*") - sum(col("has_demand"))).alias("negative_cases")
).show()

+--------------+--------------+
|positive_cases|negative_cases|
+--------------+--------------+
|         96156|       2113747|
+--------------+--------------+



In [ ]:
train_cls = train_cls.cache()
train_cls.count()  # force cache

train_cls.cache()
val_cls.cache()
test_cls.cache()

26/04/05 20:53:44 WARN CacheManager: Asked to cache already cached data.
26/04/05 20:53:45 WARN CacheManager: Asked to cache already cached data.
26/04/05 20:53:45 WARN CacheManager: Asked to cache already cached data.
26/04/05 20:53:45 WARN CacheManager: Asked to cache already cached data.


DataFrame[lat_bin: bigint, lon_bin: bigint, timestamp: timestamp, pickup_date: date, pickup_hour: int, demand: bigint, avg_fare: double, avg_distance: double, long_trip_ratio: double, lag_1: bigint, lag_2: bigint, lag_3: bigint, rolling_3: double, hour: int, day_of_week: int, month: int, hour_sin: double, hour_cos: double, loc_avg_demand: double, loc_std_demand: double, is_peak_hour: int, is_weekend: int, is_night: int, demand_vs_avg: double, demand_zscore: double, rolling_6: double, lag_24: bigint, lag_168: bigint, trend_1_3: bigint, trend_1_6: double, is_active_zone: int, peak_intensity: double, log_lag_1: double, log_rolling_6: double, hour_x_zone: double, recent_activity: double, log_demand: double, persistence: int, has_demand: int, has_demand: int, weight: double]

In [ ]:
df_model_balanced.select("demand").summary().show()

+-------+------------------+
|summary|            demand|
+-------+------------------+
|  count|           3388585|
|   mean|12.801612767571125|
| stddev| 89.31905842994387|
|    min|                 0|
|    25%|                 0|
|    50%|                 0|
|    75%|                 0|
|    max|              2658|
+-------+------------------+



In [ ]:
df_model_balanced.filter(col("demand") == 0).count()

2921510

In [ ]:
df_model_balanced.select("demand", "has_demand").show(20)

+------+----------+
|demand|has_demand|
+------+----------+
|     1|         1|
|     1|         1|
|     1|         1|
|     1|         1|
|     1|         1|
|     1|         1|
|     1|         1|
|     1|         1|
|     1|         1|
|     1|         1|
|     1|         1|
|     1|         1|
|     1|         1|
|     1|         1|
|     1|         1|
|     1|         1|
|     1|         1|
|     1|         1|
|     1|         1|
|     1|         1|
+------+----------+
only showing top 20 rows



In [ ]:
total = train_cls.count()

train_cls.groupBy("has_demand") \
    .count() \
    .withColumn("ratio", col("count") / total) \
    .show()

+----------+-------+-------------------+
|has_demand|  count|              ratio|
+----------+-------+-------------------+
|         1|  96156|0.04351141203935195|
|         0|2113747| 0.9564885879606481|
+----------+-------+-------------------+



In [ ]:
neg = 7134482
pos = 124354

ratio = neg / pos   # ≈ 57

In [ ]:
from pyspark.sql.functions import when

train_cls = train_cls.withColumn(
    "weight",
    when(col("has_demand") == 1, ratio).otherwise(1.0)
)

In [ ]:
from pyspark.ml.feature import VectorAssembler

# Use cls_features, which excludes unsupported types
# Remove unsupported types for VectorAssembler
supported_features = [c for c in cls_features if c not in ("timestamp", "pickup_date")]

assembler = VectorAssembler(
    inputCols=supported_features,
    outputCol="features"
)

train_cls_vec = assembler.transform(train_cls)
val_cls_vec   = assembler.transform(val_cls)
test_cls_vec  = assembler.transform(test_cls)

In [ ]:
train_cls_small = train_cls.sample(fraction=0.3, seed=42)
train_cls_vec = assembler.transform(train_cls_small)

In [ ]:
print([c for c in train_cls_vec.columns if c == "has_demand"])

['has_demand', 'has_demand']


In [ ]:
train_cls.columns

['lat_bin',
 'lon_bin',
 'timestamp',
 'pickup_date',
 'pickup_hour',
 'demand',
 'avg_fare',
 'avg_distance',
 'long_trip_ratio',
 'lag_1',
 'lag_2',
 'lag_3',
 'rolling_3',
 'hour',
 'day_of_week',
 'month',
 'hour_sin',
 'hour_cos',
 'loc_avg_demand',
 'loc_std_demand',
 'is_peak_hour',
 'is_weekend',
 'is_night',
 'demand_vs_avg',
 'demand_zscore',
 'rolling_6',
 'lag_24',
 'lag_168',
 'trend_1_3',
 'trend_1_6',
 'is_active_zone',
 'peak_intensity',
 'log_lag_1',
 'log_rolling_6',
 'hour_x_zone',
 'recent_activity',
 'log_demand',
 'persistence',
 'has_demand',
 'has_demand',
 'weight']

In [ ]:
train_cls_vec = train_cls_vec.selectExpr(
    "*",
    "`has_demand` as label"
).drop("has_demand")

In [ ]:
print("has_demand in cls_features?", "has_demand" in cls_features)

has_demand in cls_features? True


In [ ]:
train_cls = train_df.select(*cls_features, "has_demand", "weight")
val_cls   = val_df.select(*cls_features, "has_demand", "weight")
test_cls  = test_df.select(*cls_features, "has_demand", "weight")

In [ ]:
from collections import Counter
print([col for col, count in Counter(train_cls.columns).items() if count > 1])

['has_demand']


In [ ]:
print("has_demand in assembler input?", "has_demand" in cls_features)

has_demand in assembler input? True


In [ ]:
cls_features = [c for c in cls_features if c != "has_demand"]

In [ ]:
print("has_demand in assembler input?", "has_demand" in cls_features)

has_demand in assembler input? False


In [ ]:
train_cls = train_df.select(*cls_features, "has_demand", "weight")
val_cls   = val_df.select(*cls_features, "has_demand", "weight")
test_cls  = test_df.select(*cls_features, "has_demand", "weight")

In [ ]:
cls_features = [
    c for c in cls_features
    if c not in ["timestamp", "pickup_date"]
]

In [ ]:
train_cls.printSchema()

root
 |-- lat_bin: long (nullable = true)
 |-- lon_bin: long (nullable = true)
 |-- timestamp: timestamp (nullable = false)
 |-- pickup_date: date (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- demand: long (nullable = false)
 |-- avg_fare: double (nullable = true)
 |-- avg_distance: double (nullable = true)
 |-- long_trip_ratio: double (nullable = true)
 |-- lag_1: long (nullable = false)
 |-- lag_2: long (nullable = false)
 |-- lag_3: long (nullable = false)
 |-- rolling_3: double (nullable = false)
 |-- hour: integer (nullable = false)
 |-- day_of_week: integer (nullable = false)
 |-- month: integer (nullable = false)
 |-- hour_sin: double (nullable = true)
 |-- hour_cos: double (nullable = true)
 |-- loc_avg_demand: double (nullable = false)
 |-- loc_std_demand: double (nullable = false)
 |-- is_peak_hour: integer (nullable = false)
 |-- is_weekend: integer (nullable = false)
 |-- is_night: integer (nullable = false)
 |-- demand_vs_avg: double (nullable = true)


In [ ]:
from pyspark.sql.types import NumericType

bad_cols = [
    f.name for f in train_cls.schema.fields
    if not isinstance(f.dataType, NumericType) and f.name not in ["has_demand", "weight"]
]

print("Non-numeric columns:", bad_cols)

Non-numeric columns: ['timestamp', 'pickup_date']


In [ ]:
cls_features = [c for c in cls_features if c not in bad_cols]

In [ ]:
train_cls = train_cls.fillna(0)
val_cls   = val_cls.fillna(0)
test_cls  = test_cls.fillna(0)

In [ ]:
train_cls_vec.select("features").show(5)

+--------------------+
|            features|
+--------------------+
|[4063.0,-7402.0,9...|
|(37,[0,1,2,3,4,5,...|
|(37,[0,1,2,3,4,5,...|
|(37,[0,1,2,3,4,5,...|
|[4063.0,-7402.0,3...|
+--------------------+
only showing top 5 rows



In [ ]:
assembler = VectorAssembler(
    inputCols=cls_features,
    outputCol="features"
)

train_cls_vec = assembler.transform(train_cls)

In [ ]:
train_cls_vec.select("features", "has_demand", "weight").show(5)

+--------------------+----------+------+
|            features|has_demand|weight|
+--------------------+----------+------+
|(36,[0,1,2,3,4,5,...|         1|   2.0|
|(36,[0,1,2,3,4,5,...|         1|   2.0|
|[4063.0,-7402.0,7...|         1|   2.0|
|[4063.0,-7402.0,9...|         1|   2.0|
|[4063.0,-7402.0,1...|         1|   2.0|
+--------------------+----------+------+
only showing top 5 rows



In [ ]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="has_demand",
    numTrees=20,
    maxDepth=10,
    weightCol="weight"
)

model = rf.fit(train_cls_vec)

In [ ]:
val_preds = model.transform(val_cls_vec)

In [ ]:
# Show what we have
print("Available columns:")
print(val_preds.columns)
print("\nData types:")
val_preds.printSchema()
print("\nSample data:")
val_preds.select("has_demand", "prediction", "probability", "rawPrediction").show(5, truncate=False)

Available columns:
['lat_bin', 'lon_bin', 'timestamp', 'pickup_date', 'pickup_hour', 'demand', 'avg_fare', 'avg_distance', 'long_trip_ratio', 'lag_1', 'lag_2', 'lag_3', 'rolling_3', 'hour', 'day_of_week', 'month', 'hour_sin', 'hour_cos', 'loc_avg_demand', 'loc_std_demand', 'is_peak_hour', 'is_weekend', 'is_night', 'demand_vs_avg', 'demand_zscore', 'rolling_6', 'lag_24', 'lag_168', 'trend_1_3', 'trend_1_6', 'is_active_zone', 'peak_intensity', 'log_lag_1', 'log_rolling_6', 'hour_x_zone', 'recent_activity', 'log_demand', 'persistence', 'has_demand', 'has_demand', 'weight', 'features', 'rawPrediction', 'probability', 'prediction']

Data types:
root
 |-- lat_bin: long (nullable = true)
 |-- lon_bin: long (nullable = true)
 |-- timestamp: timestamp (nullable = false)
 |-- pickup_date: date (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- demand: long (nullable = false)
 |-- avg_fare: double (nullable = true)
 |-- avg_distance: double (nullable = true)
 |-- long_trip_ratio: 

+----------+----------+-----------+-------------+
|has_demand|prediction|probability|rawPrediction|
+----------+----------+-----------+-------------+
|1         |1.0       |[0.0,1.0]  |[0.0,20.0]   |
|1         |1.0       |[0.0,1.0]  |[0.0,20.0]   |
|1         |1.0       |[0.0,1.0]  |[0.0,20.0]   |
|1         |1.0       |[0.0,1.0]  |[0.0,20.0]   |
|1         |1.0       |[0.0,1.0]  |[0.0,20.0]   |
+----------+----------+-----------+-------------+
only showing top 5 rows



In [ ]:
# Check validation data for nulls BEFORE transformation
print("Checking for nulls in validation data:")
from pyspark.sql.functions import col, sum as spark_sum

null_counts = val_cls.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in val_cls.columns
])
null_counts.show(vertical=True)

# If you find nulls, fill them:
val_cls_clean = val_cls.fillna(0.0)  # or use appropriate strategy

# Assemble features column
val_cls_vec = assembler.transform(val_cls_clean)

# NOW transform
val_preds = model.transform(val_cls_vec)

Checking for nulls in validation data:
-RECORD 0-----------------
 lat_bin         | 0      
 lon_bin         | 0      
 timestamp       | 0      
 pickup_date     | 431547 
 pickup_hour     | 0      
 demand          | 0      
 avg_fare        | 0      
 avg_distance    | 0      
 long_trip_ratio | 0      
 lag_1           | 0      
 lag_2           | 0      
 lag_3           | 0      
 rolling_3       | 0      
 hour            | 0      
 day_of_week     | 0      
 month           | 0      
 hour_sin        | 0      
 hour_cos        | 0      
 loc_avg_demand  | 0      
 loc_std_demand  | 0      
 is_peak_hour    | 0      
 is_weekend      | 0      
 is_night        | 0      
 demand_vs_avg   | 0      
 demand_zscore   | 0      
 rolling_6       | 0      
 lag_24          | 0      
 lag_168         | 0      
 trend_1_3       | 0      
 trend_1_6       | 0      
 is_active_zone  | 0      
 peak_intensity  | 0      
 log_lag_1       | 0      
 log_rolling_6   | 0      
 hour_x_zone    

In [ ]:
# Compare training vs validation schemas
print("Training data schema:")
train_cls.printSchema()

print("\nValidation data schema:")
val_cls.printSchema()

# Look for mismatches in:
# - Column names
# - Data types (long vs double, integer vs double)
# - Nullable flags

Training data schema:
root
 |-- lat_bin: long (nullable = true)
 |-- lon_bin: long (nullable = true)
 |-- timestamp: timestamp (nullable = false)
 |-- pickup_date: date (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- demand: long (nullable = false)
 |-- avg_fare: double (nullable = false)
 |-- avg_distance: double (nullable = false)
 |-- long_trip_ratio: double (nullable = false)
 |-- lag_1: long (nullable = false)
 |-- lag_2: long (nullable = false)
 |-- lag_3: long (nullable = false)
 |-- rolling_3: double (nullable = false)
 |-- hour: integer (nullable = false)
 |-- day_of_week: integer (nullable = false)
 |-- month: integer (nullable = false)
 |-- hour_sin: double (nullable = false)
 |-- hour_cos: double (nullable = false)
 |-- loc_avg_demand: double (nullable = false)
 |-- loc_std_demand: double (nullable = false)
 |-- is_peak_hour: integer (nullable = false)
 |-- is_weekend: integer (nullable = false)
 |-- is_night: integer (nullable = false)
 |-- demand_vs_avg

In [ ]:
train_cls_vec.printSchema()

root
 |-- lat_bin: long (nullable = true)
 |-- lon_bin: long (nullable = true)
 |-- timestamp: timestamp (nullable = false)
 |-- pickup_date: date (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- demand: long (nullable = false)
 |-- avg_fare: double (nullable = false)
 |-- avg_distance: double (nullable = false)
 |-- long_trip_ratio: double (nullable = false)
 |-- lag_1: long (nullable = false)
 |-- lag_2: long (nullable = false)
 |-- lag_3: long (nullable = false)
 |-- rolling_3: double (nullable = false)
 |-- hour: integer (nullable = false)
 |-- day_of_week: integer (nullable = false)
 |-- month: integer (nullable = false)
 |-- hour_sin: double (nullable = false)
 |-- hour_cos: double (nullable = false)
 |-- loc_avg_demand: double (nullable = false)
 |-- loc_std_demand: double (nullable = false)
 |-- is_peak_hour: integer (nullable = false)
 |-- is_weekend: integer (nullable = false)
 |-- is_night: integer (nullable = false)
 |-- demand_vs_avg: double (nullable = f